In [1]:
from dotenv import load_dotenv
load_dotenv()
import uuid
from typing import List
from pydantic import BaseModel,Field
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph,START,END,MessagesState
from langgraph.store.memory import InMemoryStore

from langgraph.store.base import BaseStore

In [2]:
# 1) make a memory store 
store=InMemoryStore()

In [5]:
# 2) llm that decides what to remember
extractor_llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [ ]:
# we made a pydantic model for it
class MemoryDecision(BaseModel):
    should_write: bool = Field(description="whether to store any memories")
    memories : List[str] = Field(default_factory=list,description="atomic user memories to store")

In [6]:
memory_extractor=extractor_llm.with_structured_output(MemoryDecision)

In [7]:
# make graph start-remember-end
def remember_only_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = ("user", user_id, "user_details")
    # take latest user msg
    last_msg = state["messages"][-1].content
    # llm decides what to store
    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(
                content=(
                    "Extract the long term memories from the user's text\n"
                    "Each memory should be short and atomic"
                )
            ),
            {"role": "user", "content": last_msg},
        ]
    )
    # write to store
    if decision.should_write:
        for mem in decision.memories:
            store.put(namespace, str(uuid.uuid4()), {"data": mem})
    # we just return a fixed acknowledgement
    return {"messages": [{"role": "assistant", "content": "Noted."}]}